# Data Preprocessing for Singapore Hawker Reviews

This notebook preprocesses the `review_text` column for sentiment analysis while preserving local Singlish context.

In [11]:
import re
import pandas as pd

# Show full text width when previewing results
pd.set_option('display.max_colwidth', 160)

In [12]:
# ------------------------------
# 1) Load data
# ------------------------------
INPUT_CSV = "hawker_corpus_final10k.csv"
OUTPUT_CSV = "hawker_corpus_final10k_cleaned.csv"

df = pd.read_csv(INPUT_CSV)
print(f"Loaded rows: {len(df):,}")
print("Columns:", df.columns.tolist())

# Ensure review_text is string and handle nulls once for efficiency
df["review_text"] = df["review_text"].fillna("").astype(str)

Loaded rows: 11,032
Columns: ['hawker_centre', 'stall_name', 'review_text', 'star_rating', 'sentiment', 'word_count']


In [13]:
# ------------------------------
# Add unique Review_ID as first column
# ------------------------------
# If Review_ID already exists (rerun case), remove it first to avoid duplicate-column errors.
if "Review_ID" in df.columns:
    df = df.drop(columns=["Review_ID"])

# Create sequential unique IDs: RV00001, RV00002, ...
df.insert(0, "Review_ID", [f"RV{i:05d}" for i in range(1, len(df) + 1)])

# Save back to the original CSV (overwrite)
df.to_csv(INPUT_CSV, index=False)

print("Review_ID added and saved to", INPUT_CSV)
df.head(3)

Review_ID added and saved to hawker_corpus_final10k.csv


,Review_ID,hawker_centre,stall_name,review_text,star_rating,sentiment,word_count
0,RV00001,Adam Road Food Centre,Selera Rasa Nasi Lemak,This is definitely a 5-Stars Nasi Lemak! 🌟🌟🌟🌟🌟 Easily top of my list for my favourite Nasi Lemak in Singapore.\n\nWhat's important in a Nasi Lemak is the ch...,5,Positive,322
1,RV00002,Adam Road Food Centre,Selera Rasa Nasi Lemak,Good deal. Great service. Fragrant nasi lemak (coconut infused rice). Otah good. Fried chicken batter a bit too hard but chicken wing meat freshly good so i...,5,Positive,56
2,RV00003,Adam Road Food Centre,Selera Rasa Nasi Lemak,"Better than average nasi lemak! I had the royal rumble, which include all the ingredients, chicken, fish, otah, bergedil, egg. The stall holders were friend...",5,Positive,104


In [14]:
# ------------------------------
# 2) Compile regex patterns once (faster for 10k+ rows)
# ------------------------------
HTML_RE = re.compile(r"<[^>]+>")
URL_RE = re.compile(r"https?://\S+|www\.\S+")

# Keep letters, numbers, whitespace, apostrophe, and sentiment punctuation ! ? . ,
SPECIAL_CHAR_RE = re.compile(r"[^a-zA-Z0-9\s'!?.,]")

# Reduce elongated characters: 3+ repeated chars -> 2 chars (e.g. sooooo -> soo)
ELONGATED_RE = re.compile(r"(.)\1{2,}")

# Normalize extra spaces
MULTISPACE_RE = re.compile(r"\s+")

In [15]:
# ------------------------------
# 3) Stopwords setup (keep negations)
# ------------------------------
# Minimal English stopword list for demonstration/report use.
# You can expand this list if needed for your modeling experiments.
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "but", "by", "for", "from", "has", "have",
    "he", "her", "him", "his", "i", "in", "is", "it", "its", "of", "on", "or", "our",
    "she", "that", "the", "their", "them", "they", "this", "to", "was", "we", "were", "will",
    "with", "you", "your", "yours", "me", "my", "mine", "us", "do", "does", "did", "am", "lah", "leh", "lor"
}

# Negations are explicitly preserved for sentiment polarity.
NEGATIONS_TO_KEEP = {"not", "never", "no", "n't"}
STOPWORDS = STOPWORDS - NEGATIONS_TO_KEEP

# Local terms are preserved naturally because we do not autocorrect or strip alphabetic tokens.
LOCAL_TERMS_EXAMPLES = {"shiok", "dabao", "kopitiam", "waseh"}

In [16]:
# ------------------------------
# 4) Preprocessing functions
# ------------------------------
def clean_text(text: str) -> str:
    """
    Remove noise, normalize microtext, and lowercase.
    Keeps basic punctuation (! ? . ,) for sentiment clues.
    """
    text = HTML_RE.sub(" ", text)
    text = URL_RE.sub(" ", text)
    text = SPECIAL_CHAR_RE.sub(" ", text)
    text = ELONGATED_RE.sub(r"\1\1", text)
    text = text.lower()
    text = MULTISPACE_RE.sub(" ", text).strip()
    return text

def tokenize_and_remove_stopwords(text: str) -> str:
    """
    Tokenize text and remove stopwords while keeping negations.
    Returns a whitespace-joined token string in cleaned form.
    """
    # Tokenization keeps words and selected punctuation as separate tokens.
    tokens = re.findall(r"[a-zA-Z]+(?:'[a-zA-Z]+)?|[!?.,]", text)

    filtered_tokens = [
        tok for tok in tokens
        if tok in {"!", "?", ".", ","} or tok not in STOPWORDS
    ]

    return " ".join(filtered_tokens)

In [17]:
# ------------------------------
# 5) Apply preprocessing pipeline
# ------------------------------
# Vectorized pandas string operation for cleaning + .apply for token filtering
# This is efficient and suitable for 10,000+ rows.
cleaned_base = df["review_text"].map(clean_text)
df["cleaned_review"] = cleaned_base.map(tokenize_and_remove_stopwords)

df[["review_text", "cleaned_review"]].head(10)

,review_text,cleaned_review
0,This is definitely a 5-Stars Nasi Lemak! 🌟🌟🌟🌟🌟 Easily top of my list for my favourite Nasi Lemak in Singapore.\n\nWhat's important in a Nasi Lemak is the ch...,definitely stars nasi lemak ! easily top list favourite nasi lemak singapore . what's important nasi lemak chilli . chilli very nice well balanced right amo...
1,Good deal. Great service. Fragrant nasi lemak (coconut infused rice). Otah good. Fried chicken batter a bit too hard but chicken wing meat freshly good so i...,good deal . great service . fragrant nasi lemak coconut infused rice . otah good . fried chicken batter bit too hard chicken wing meat freshly good so tende...
2,"Better than average nasi lemak! I had the royal rumble, which include all the ingredients, chicken, fish, otah, bergedil, egg. The stall holders were friend...","better than average nasi lemak ! had royal rumble , which include all ingredients , chicken , fish , otah , bergedil , egg . stall holders friendly well , s..."
3,"So many halal stall still open, time now is 9pm on friday night.\nAlso the food that i bought mutton ribs soup, taste soo delicious.\nAnd youbadd on some w...","so many halal stall still open , time now pm friday night . also food bought mutton ribs soup , taste soo delicious . youbadd some white pepper powder soy s..."
4,"Nice, clean and airy food court. Food is majority Malay/Muslim/Mamak. Little variety; many stalls offer similar items; nasi lemak, fried rice and fried nood...","nice , clean airy food court . food majority malay muslim mamak . little variety many stalls offer similar items nasi lemak , fried rice fried noodles . wel..."
5,Definitely one of the older Nasi Lemak stalls in Adam Road Food Center. I decided to try it out instead of Selera Rasa because of the super long queue. The ...,definitely one older nasi lemak stalls adam road food center . decided try out instead selera rasa because super long queue . makcik also very friendly . ho...
6,"Horrendous! Utterly unethical. Bought 2 Chicken Wing Set for family for lunch as today is CNY, am deeply disgusted and utterly pissed off.\n\nThey have the ...","horrendous ! utterly unethical . bought chicken wing set family lunch today cny , deeply disgusted utterly pissed off . audacity selling uncooked coconut ri..."
7,"Honestly, I feel that this store deserves more traction, i have tried all the other nasi lemaks at this hawker centre but find this was the stand out. The f...","honestly , feel store deserves more traction , tried all other nasi lemaks hawker centre find stand out . fried chicken flavourful hot , rice fragrant , sam..."
8,The nasi lemak is very above average cos the rice is very lemak. Fragrant but a bit inconsistent so you go on different days and it doesn’t taste the same. ...,nasi lemak very above average cos rice very lemak . fragrant bit inconsistent so go different days doesn t taste same . fried chicken decent enough . bigges...
9,Always go to this Nasi Lemak rather than the one (I had it once it was horrible - the egg was cold and oily and the otak already sour) beside it which spots...,always go nasi lemak rather than one had once horrible egg cold oily otak already sour beside which spots super long queue . understand crowd effect everyon...


In [18]:
# ------------------------------
# 6) Save output
# ------------------------------
df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved cleaned dataset to: {OUTPUT_CSV}")
print("Output shape:", df.shape)

Saved cleaned dataset to: hawker_corpus_final10k_cleaned.csv
Output shape: (11032, 8)


In [19]:
# ------------------------------
# Manual labeling task prep (Excel)
# ------------------------------
import pandas as pd

INPUT_CSV = "hawker_corpus_final10k.csv"
OUTPUT_XLSX = "Hand_Labeling_Task.xlsx"

df = pd.read_csv(INPUT_CSV)

# Fallback: if Review_ID is missing, create it (keeps workflow robust)
if "Review_ID" not in df.columns:
    df.insert(0, "Review_ID", [f"RV{i:05d}" for i in range(1, len(df) + 1)])

# Randomly sample 1,000 records for labeling
if len(df) < 1000:
    raise ValueError(f"Dataset has only {len(df)} rows; need at least 1000 for sampling.")

sampled = df.sample(n=1000, random_state=42).copy()

# Create the Excel table with EXACTLY the required columns
# Subjectivity and Polarity are empty (blank) for manual labeling.
label_df = pd.DataFrame({
    "Review_ID": sampled["Review_ID"],
    "Hawker_Centre": sampled["hawker_centre"],
    "Stall_Name": sampled["stall_name"],
    "Review_Text": sampled["review_text"],  # original uncleaned text
    "Subjectivity": [pd.NA] * len(sampled),
    "Polarity": [pd.NA] * len(sampled),
})

# Export to Excel
label_df.to_excel(OUTPUT_XLSX, index=False)
print(f"Saved manual labeling file to: {OUTPUT_XLSX}")
print("Output columns:", label_df.columns.tolist())
label_df.head(3)

Saved manual labeling file to: Hand_Labeling_Task.xlsx
Output columns: ['Review_ID', 'Hawker_Centre', 'Stall_Name', 'Review_Text', 'Subjectivity', 'Polarity']


,Review_ID,Hawker_Centre,Stall_Name,Review_Text,Subjectivity,Polarity
6982,RV06983,Blk 44 Holland Drive,Al Fatimah,"Happen to be seated outside Al Fatimah, the pratas looked quite fluffy and decided to give one a try despite already having breakfast, indeed the prata are ...",<NA>,<NA>
1597,RV01598,Kallang Estate Market,Ah Teck,Fried bee hoon and mee (vermicelli and noodle) is one of those things that locals like to eat in the morning. To me the key of this dish is the way and how ...,<NA>,<NA>
4919,RV04920,Blk 16 Bedok South Road,Tian Tian Xiang Shang Grilled Fish,Food is good. Sometimes a little inconsistent. But generally good and pretty cheap. Gone back there again and again. The lady doesn't have the best of attit...,<NA>,<NA>
